In [ ]:
!git clone https://github.com/DimitrisKu/Active-Reading--Pattern-Recognition.git

import os

%cd /content/Active-Reading--Pattern-Recognition

os.getcwd()

In [ ]:
# =========================
# Imports & setup
# =========================
# !pip -q install transformers
import json
import re
from pathlib import Path
from transformers import AutoTokenizer

# -------------------------
# Paths
# -------------------------
BASE_DIR = Path("/content/Active-Reading--Pattern-Recognition")

ORIGINAL_PATH = BASE_DIR / "Datasets" / "finance_bench_corpus.json"   #    "simple_wiki_corpus.json"
PARA_DIR      = BASE_DIR  / "paraphrase_outputs"        # / "generated_simplewiki"
QA_DIR        = BASE_DIR /  "synthetic_qa_outputs"
ACT_READ_DIR  = BASE_DIR /  "active_reading_outputs"

# -------------------------
# Tokenizer
# -------------------------
MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)

def n_tokens(text: str) -> int:
    return len(tokenizer.encode(text, add_special_tokens=False)) if text else 0

# -------------------------
# Word counter
# -------------------------
WORD_RE = re.compile(r"\b[\w’']+\b", re.UNICODE)

def n_words(text: str) -> int:
    return len(WORD_RE.findall(text)) if text else 0

In [27]:
def count_original_dataset(json_path: Path, max_rows: int | None = None):
    rows = files = tokens = words = 0
    files = 1  # single JSON file
    SKIP_DOCS = {"MICROSOFT_2016_10K"}   # skip this doc for financebench corpus

    with json_path.open("r", encoding="utf-8") as f:
        data = json.load(f)

    # Sort deterministically by doc_name (missing values go last)
    data = sorted(data, key=lambda obj: (obj.get("doc_name") is None, obj.get("doc_name")))

    data = [
        obj for obj in data
        if obj.get("doc_name") not in SKIP_DOCS
    ]

    for obj in data:
        if max_rows is not None and rows >= max_rows:
            break

        text = (obj.get("text") or "").strip()
        if not text:
            continue

        rows += 1
        tokens += n_tokens(text)
        words  += n_words(text)

    return {
        "files": files,
        "rows": rows,
        "tokens": tokens,
        "words": words
    }

In [28]:
def count_jsonl_folder(folder: Path, fields, max_files: int | None = None):
    rows = tokens = words = 0
    SKIP_FILES = {"MICROSOFT_2016_10K.jsonl"}   # skip this file for financebench dataset

    files = sorted(folder.glob("*.jsonl"))
    files = [
        p for p in files
        if p.name not in SKIP_FILES
    ]

    if max_files is not None:
        files = files[:max_files]

    for path in files:
        with path.open("r", encoding="utf-8") as f:
            for line in f:
                try:
                    obj = json.loads(line)
                except json.JSONDecodeError:
                    continue

                text = " ".join((obj.get(k) or "").strip() for k in fields).strip()

                if not text:
                    continue

                rows += 1
                tokens += n_tokens(text)
                words  += n_words(text)

    return {
        "files": len(files),
        "rows": rows,
        "tokens": tokens,
        "words": words
    }

In [29]:
# Number of files to take into account. If None, then count all
ORG_ROWS  = 20
PARA_FILES = 20
QA_FILES   = 20
AR_FILES   = 20

In [30]:
original_stats = count_original_dataset(ORIGINAL_PATH, max_rows=ORG_ROWS)

para_stats = count_jsonl_folder(PARA_DIR, fields=("text",), max_files=PARA_FILES)

qa_stats = count_jsonl_folder(QA_DIR, fields=("question", "answer"), max_files=QA_FILES)

act_read_stats = count_jsonl_folder(ACT_READ_DIR, fields=("active_reading",),  # or ("strategy","active_reading")
    max_files=AR_FILES)

In [ ]:
def show(name, stats):
    print(f"{name:<14} | "
          f"files={stats['files']:>3} | "
          f"rows={stats['rows']:>8,} | "
          f"tokens={stats['tokens']:>12,} | "
          f"words={stats['words']:>12,}")

print("\n=== DATASET STATISTICS ===")
show("Original", original_stats)
show("Paraphrase", para_stats)
show("QA", qa_stats)
show("ActiveReading", act_read_stats)

In [ ]:
gen_total_tokens = para_stats["tokens"] + qa_stats["tokens"] + act_read_stats["tokens"]

if gen_total_tokens > 0:
    print("\n=== TOKEN SHARE (Generated) ===")
    print(f"Paraphrase: {para_stats['tokens'] / gen_total_tokens * 100:6.2f}%")
    print(f"QA:         {qa_stats['tokens'] / gen_total_tokens * 100:6.2f}%")
    print(f"ActiveReading: {act_read_stats['tokens'] / gen_total_tokens * 100:6.2f}%")

# Code to subsample QA and Active Reading generated tokens to match Paraphrase

In [ ]:
import json
import random
from pathlib import Path
from transformers import AutoTokenizer


# example for QA subsampling, same logic for AR

SRC_DIR = Path("synthetic_qa_outputs")
OUT_DIR = Path("synthetic_qa_outputs_budgeted")
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def qa_to_text(q, a):
    return q.strip() + "\n" + a.strip() + tokenizer.eos_token

def count_tokens(text: str) -> int:
    return len(tokenizer(text, add_special_tokens=False)["input_ids"])

def load_all_rows(jsonl_paths):
    rows = []
    for p in jsonl_paths:
        with open(p, "r", encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue
                ex = json.loads(line)
                # Expect fields: question, answer
                if "question" in ex and "answer" in ex:
                    rows.append(ex)
    return rows

# 1) Read all QA rows
qa_files = sorted(SRC_DIR.glob("*.jsonl"))
all_rows = load_all_rows(qa_files)
print("Loaded QA pairs:", len(all_rows))

# 2) Shuffle
random.shuffle(all_rows)

# 3) Greedily keep until token budget N
# Set this to your paraphrase token count:
N =  para_stats['tokens']

kept = []
total = 0
for ex in all_rows:
    txt = qa_to_text(ex["question"], ex["answer"])
    t = count_tokens(txt)
    if total + t > N:
        continue
    kept.append(ex)
    total += t
    if total >= N:
        break

print(f"Kept QA pairs: {len(kept)}")
print(f"Total QA tokens: {total} (target N={N})")

# 4) Write as one JSONL (simplest)
out_path = OUT_DIR / "qa_budgeted.jsonl"
with open(out_path, "w", encoding="utf-8") as f:
    for ex in kept:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")

print("Wrote:", out_path)